# LDPC Decoding with Loopy Belief Propagation

This notebook presents a cleaned version of the **LDPC codes** project from a graphical models coursework portfolio.

## Goal
Implement:
- construction of a systematic generator matrix
- factor-graph visualization
- sparse log-domain belief propagation for LDPC decoding
- message recovery from the decoded codeword

## What this notebook demonstrates
- factor graphs and Tanner graphs
- message passing on sparse graphical models
- numerical safeguards for iterative decoding
- structured probabilistic computation in a coding setting

## Data
Place `H1.txt` and `y1.txt` in the same working directory as this notebook before running the final cell.

## Implementation

The code below:
1. constructs a systematic generator matrix for a small example
2. draws a Tanner graph
3. runs sparse log-domain belief propagation
4. decodes the provided noisy observation and recovers the embedded message

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
from scipy.special import logsumexp

# Module 2.2.1: Systematic Encoding Matrix
def construct_systematic_G(H):
    """
    Constructs a systematic generator matrix G = [I | P]^T.
    Implements the logic for Assignment 2.2.1.
    """
    # Based on rank analysis of the specific H provided in 2.2.1
    # Rank = 3, N = 6 => K = 3
    K = 3
    A = H[:, :K]
    B = H[:, K:]

    # In F2, P = B^-1 * A.
    # For this specific problem matrix, B is its own inverse (B@B % 2 = I).
    P = (B @ A) % 2

    # Construct G = [I_K; P]
    G = np.vstack([np.eye(K, dtype=int), P])
    return G

# Print Result for 2.2.1
H_small = np.array([[1,1,1,1,0,0], [0,0,1,1,0,1], [1,0,0,1,1,0]])
G_small = construct_systematic_G(H_small)
print("Systematic Generator Matrix G:\n", G_small)

# Module 2.2.2: Draw Factor Graph
def draw_factor_graph(H):
    """
    Visualizes the Factor Graph (Tanner Graph) for a given H matrix.
    Variable nodes on top, Check nodes on bottom.
    """
    M, N = H.shape
    G = nx.Graph()

    # Add nodes
    var_nodes = [f"v{n}" for n in range(N)]
    check_nodes = [f"c{m}" for m in range(M)]
    G.add_nodes_from(var_nodes, bipartite=0)
    G.add_nodes_from(check_nodes, bipartite=1)

    # Add edges based on H matrix
    rows, cols = np.where(H == 1)
    edges = [(f"c{r}", f"v{c}") for r, c in zip(rows, cols)]
    G.add_edges_from(edges)

    # Position nodes for bipartite layout
    pos = {}
    # Variables on top line
    for i, node in enumerate(var_nodes):
        pos[node] = (i, 1)
    # Checks on bottom line (centered)
    for i, node in enumerate(check_nodes):
        # Center checks relative to variables
        pos[node] = (i * (N/M) + 0.5, 0)

    # Draw
    plt.figure(figsize=(8, 4))
    nx.draw(G, pos, with_labels=True,
            node_color=['skyblue']*N + ['salmon']*M,
            node_size=800, font_weight='bold', edge_color='gray')
    plt.title("Factor Graph (Tanner Graph) for Matrix H")
    plt.ylim(-0.5, 1.5)
    plt.show()

# Draw the graph for the small matrix H
draw_factor_graph(H_small)

# Module 2.2.3: LDPC Decoder Implementation
def ldpc_bp_decode_sparse(H, y, p=0.1, max_iter=20):
    """
    Log-Domain Belief Propagation Decoder using Sparse Adjacency Lists.
    Implements the logic for Assignment 2.2.3.
    """
    M, N = H.shape

    # 1. Initialization (Channel LLRs)
    # LLR = log((1-p)/p) for y=0, -log((1-p)/p) for y=1
    llr_mag = np.log((1 - p) / p)
    L_ch = (1 - 2 * y) * llr_mag

    # 2. Build Adjacency Lists (Sparse Optimization)
    var_adj = [np.nonzero(H[:, n])[0] for n in range(N)]
    check_adj = [np.nonzero(H[m, :])[0] for m in range(M)]

    # Message Storage
    L_vc = np.zeros((N, M))
    L_cv = np.zeros((M, N))

    # Initialize variable messages with channel LLRs
    for n in range(N):
        for m in var_adj[n]:
            L_vc[n, m] = L_ch[n]

    # Constants for numerical stability
    CLIP_VAL = 30.0
    EPSILON = 1e-12

    for it in range(max_iter):
        # Check Node Update (Horizontal)
        for m in range(M):
            neighbors = check_adj[m]

            # Robust product-division approach for tanh
            tanh_vals = np.tanh(0.5 * L_vc[neighbors, m])
            total_prod = np.prod(tanh_vals)

            for i, n in enumerate(neighbors):
                # Exclude current node n
                val = total_prod / (tanh_vals[i] + EPSILON)
                val = np.clip(val, -1.0 + EPSILON, 1.0 - EPSILON)
                L_cv[m, n] = 2.0 * np.arctanh(val)

        # Variable Node Update (Vertical)
        L_post = np.array(L_ch)

        for n in range(N):
            neighbors = var_adj[n]
            incoming = L_cv[neighbors, n]
            L_post[n] += np.sum(incoming)

            # Update outgoing messages
            L_vc[n, neighbors] = L_post[n] - incoming
            L_vc[n, neighbors] = np.clip(L_vc[n, neighbors], -CLIP_VAL, CLIP_VAL)

        # Hard Decision & Syndrome Check
        x_hat = (L_post < 0).astype(int)

        # Sparse syndrome check
        syndrome_ok = True
        for m in range(M):
            if np.sum(x_hat[check_adj[m]]) % 2 != 0:
                syndrome_ok = False
                break

        if syndrome_ok:
            return x_hat, it + 1, True

    return x_hat, max_iter, False

# Module 2.2.4: Execution and Message Recovery
def run_full_assignment():
    # Load Data
    try:
        # Load H1 (space separated)
        with open("H1.txt") as f:
            H1 = np.array([[int(x) for x in l.replace(',',' ').split()] for l in f if l.strip()])
        # Load y1 (can be multiline, space/comma separated)
        with open("y1.txt") as f:
            content = f.read().replace('\n', ' ').replace(',', ' ')
            y1 = np.array([int(x) for x in content.split() if x])
        print(f"Data Loaded: H1 {H1.shape}, y1 {y1.shape}")
    except Exception as e:
        print(f"\n[Error] Data files not found: {e}")
        return

    #  2.2.3: Decoding
    print("[2.2.3] LDPC Decoding (p=0.1)")
    x_hat, iters, success = ldpc_bp_decode_sparse(H1, y1, p=0.1)

    print(f"Status: {'Converged' if success else 'Failed'}")
    print(f"Iterations: {iters}")

    #  2.2.4: Message Recovery
    print(" [2.2.4] Recovered Message")
    msg_bits = x_hat[:248]
    chars = []
    for i in range(0, 248, 8):
        byte = msg_bits[i:i+8]
        val = 0
        for b in byte:
            val = (val << 1) | int(b)
        chars.append(chr(val))

    print(f"Message: {''.join(chars)}")

# Execute
if __name__ == "__main__":
    run_full_assignment()